# Graficas en orden de aparición en el index.qmd final

## Texto introductorio al tablero:

¿Qué es la Deuda Hídrica?

La deuda hídrica no es una metáfora; es un déficit contable y ecológico real. Ocurre cuando el agua que extraemos para nuestras actividades supera la capacidad de la naturaleza para renovarse. A nivel individual, es un préstamo invisible: cada litro que consumimos hoy bajo este esquema es agua que las futuras generaciones ya no podrán utilizar.

# ¿Cuánta agua hay?

Texto introductorio: 

En México, la crisis hídrica no se explica únicamente por la falta de agua a nivel nacional, sino por una profunda desconexión entre la geografía del recurso y la distribución de la población.

Mientras que las zonas del sur del país, como la región de la Frontera Sur (con estados como Chiapas, Oaxaca y Tabasco), concentran la mayor abundancia de agua renovable per cápita , los polos de mayor densidad demográfica y económica del centro y norte —como el Valle de México— operan con una disponibilidad por habitante críticamente baja.

Esta presión poblacional sobre territorios con estrés hídrico obliga a los gobiernos locales a administrar la urgencia en lugar de planificar el futuro. Al observar la respuesta de las autoridades ante los asentamientos urbanos, es evidente que el esfuerzo es mayoritariamente reactivo: más de la mitad de las acciones municipales (53.9%) se destinan a intentar dotar de servicios básicos a zonas que muchas veces crecieron sin planeación. Por el contrario, las soluciones estructurales de fondo, como la regularización (6.78%) o la reubicación (9.49%), ocupan un porcentaje mínimo de la agenda pública.

La verdadera "deuda líquida" radica aquí: estamos forzando el crecimiento urbano en regiones donde la naturaleza hace tiempo dejó de tener crédito, mientras los municipios luchan por llevar tuberías a lugares donde, geográficamente, los números ya no cuadran.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import re
import os
from plotly.subplots import make_subplots
import numpy as np

In [2]:
# Poblacion vs recursos 2021

#  Datos
files_agua = [
    "../datos/AguaRenovable/EntidadFederativa/Agua_Renovable_EntidadFederativa_2017.xlsx",
    "../datos/AguaRenovable/EntidadFederativa/Agua_Renovable_EntidadFederativa_2018.xlsx",
    "../datos/AguaRenovable/EntidadFederativa/Agua_Renovable_EntidadFederativa_2019.xlsx",
    "../datos/AguaRenovable/EntidadFederativa/Agua_Renovable_EntidadFederativa_2020.xlsx",
    "../datos/AguaRenovable/EntidadFederativa/Agua_Renovable_EntidadFederativa_2021.xlsx"
]

all_data_agua = []
for f in files_agua:
    if os.path.exists(f):
        df_temp = pd.read_excel(f, engine='openpyxl')
        all_data_agua.append(df_temp)

master_agua = pd.concat(all_data_agua, ignore_index=True)
df_2021 = master_agua[master_agua['Año'] == 2021]
# Limpieza de columnas numéricas
cols_num = ['Agua Renovable (hm³/año)', 'Población (Millones de habitantes)', 'Agua Renovable per cápita (m³/habitante/año)']
for col in cols_num:
    master_agua[col] = pd.to_numeric(master_agua[col].astype(str).str.replace(',', '').str.replace('"', '').str.strip(), errors='coerce').fillna(0)

# Población vs Recursos
fig4 = px.scatter(df_2021, x='Población (Millones de habitantes)', y='Agua Renovable (hm³/año)',
                 size='Agua Renovable per cápita (m³/habitante/año)', color='Entidad Federativa',
                 title='<b>Población vs. Recursos (2021)</b>')
fig4.update_layout(height=350, margin=dict(l=10, r=10, t=50, b=10), showlegend=False)

fig4.show()

In [3]:
# Poblacion vs concesiones
# NOTA: Se uso proyeccion de datos para las poblaciones porque el inegi solo cuenta con informacion del 210 y 2020
# 1. CARGA DE DATOS DE AGUA (6.4.2)
path_agua = '../datos/6N.2.1_es/conjunto_de_datos/6n.2.1_dc_1215_es.csv'
df_ind = pd.read_csv(path_agua, encoding='latin1', sep=None, engine='python')
df_ind.columns = [c.strip() for c in df_ind.columns]
df_ind = df_ind.iloc[:, [0, 1]].dropna()
df_ind.columns = ['Anio', 'Volumen_km3']

# 2. CARGA DE POBLACIÓN (Búsqueda dinámica en el Excel)
path_pop = '../datos/Poblacion/Poblacion_01-2.xlsx'
df_pop_raw = pd.read_excel(path_pop, sheet_name='Tabulado', header=None)

# Localizamos la fila de "Estados Unidos Mexicanos"
mask = df_pop_raw.apply(lambda row: row.astype(str).str.contains("Estados Unidos Mexicanos", case=False, na=False).any(), axis=1)
fila_nacional = df_pop_raw[mask]

if not fila_nacional.empty:
    def clean_num(val):
        return float(str(val).replace('\xa0', '').replace(' ', '').replace(',', ''))

    # Totales de 2010 (Col 5) y 2020 (Col 8) para la tendencia
    pop_2010 = clean_num(fila_nacional.iloc[0, 5])
    pop_2020 = clean_num(fila_nacional.iloc[0, 8])

    # 3. PROYECCIÓN DESDE 2012
    tasa_crecimiento = (pop_2020 - pop_2010) / 10
    años_grafica = list(range(2012, int(df_ind['Anio'].max()) + 1))
    df_pob = pd.DataFrame({'Anio': años_grafica})
    df_pob['Poblacion'] = pop_2010 + (df_pob['Anio'] - 2010) * tasa_crecimiento

    # Filtramos agua para empezar en 2012
    df_ind = df_ind[df_ind['Anio'] >= 2012].sort_values('Anio')

    # 4. CREACIÓN DE LA GRÁFICA
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Fondo: Población (Eje Y Secundario)
    fig.add_trace(
        go.Scatter(
            x=df_pob['Anio'], y=df_pob['Poblacion'] / 1e6,
            fill='tozeroy', name='Población (Millones)',
            fillcolor='rgba(184, 212, 255, 0.2)', line=dict(color='rgba(184, 212, 255, 0.4)')
        ),
        secondary_y=True
    )

    # Frente: Concesiones (Eje Y Primario)
    fig.add_trace(
        go.Scatter(
            x=df_ind['Anio'], y=df_ind['Volumen_km3'],
            mode='lines+markers', name='Agua Concesionada (km³)',
            line=dict(color='#0066FF', width=4), marker=dict(size=8)
        ),
        secondary_y=False
    )

    # 5. CONFIGURACIÓN DE EJES (INICIO EN 0)
    fig.update_layout(
        title='<b>Análisis de Deuda Hídrica (2012-2023)</b>',
        xaxis_title='Año', template='plotly_white', hovermode='x unified',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    # Eje Primario (Azul - Izquierda)
    fig.update_yaxes(
        title_text="<b>Volumen Concesionado (km³)</b>", 
        color="#0066FF", 
        secondary_y=False,
        rangemode="tozero" # <--- ESTO FUERZA EL 0
    )

    # Eje Secundario (Gris - Derecha)
    fig.update_yaxes(
        title_text="Población (Millones)", 
        secondary_y=True, 
        showgrid=False,
        rangemode="tozero" # <--- ESTO FUERZA EL 0
    )

    fig.show()

In [4]:
# Estados con mayor abundancia
abundancia = df_2021.sort_values('Agua Renovable per cápita (m³/habitante/año)', ascending=True)
fig3 = px.bar(abundancia, x='Agua Renovable per cápita (m³/habitante/año)', y='Entidad Federativa', 
             orientation='h', title='<b>Estados con Mayor Abundancia</b>',
             color='Agua Renovable per cápita (m³/habitante/año)', color_continuous_scale='Blues')
fig3.update_layout(height=350, margin=dict(l=10, r=10, t=50, b=10), coloraxis_showscale=False, yaxis_title="")

In [5]:
# Brecha de servicios
# GRÁFICA 1: Brecha de Servicios
df_serv = pd.read_csv("../datos/Res_territorial_ah_CNGMD2021/res_terr_ah_irreg_cngmd2021_csv/conjunto_de_datos/m2s13p23_cngmd2021.csv")
service_names = {1: 'Agua potable', 2: 'Drenaje', 3: 'Elect', 4: 'Alumbrado', 5: 'Pavimento', 6: 'Basura'}
df_serv['Servicio'] = df_serv['servpub_a'].map(service_names)
df_serv['Existencia'] = df_serv['rensnn1'].apply(lambda x: 1 if x == 1 else 0)

stats = df_serv.groupby('Servicio')['Existencia'].sum().reset_index()
total_mun = df_serv['ubicageo_c'].nunique() # son 2467 registrados y en realidad son 2478
stats['Falta'] = total_mun - stats['Existencia']

fig_gap = go.Figure()
fig_gap.add_trace(go.Bar(name='Con Servicio', x=stats['Servicio'], y=stats['Existencia'], marker_color='#00C8B4'))
fig_gap.add_trace(go.Bar(name='Sin Servicio', x=stats['Servicio'], y=stats['Falta'], marker_color='#FF3B3B'))

fig_gap.update_layout(
    barmode='stack', 
    title='<b>Brecha de Servicios</b>',
    xaxis_title='', 
    yaxis_title='Cantidad de Municipios', 
    template='plotly_white',
    height=350,
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_gap.show()

In [6]:
# Acciones municipales 
df_actions = pd.read_csv("../datos/Res_territorial_ah_CNGMD2021/res_terr_ah_irreg_cngmd2021_csv/conjunto_de_datos/m2s13p22_cngmd2021.csv")
act_names = {1: 'Regularización', 2: 'Dotación Servicios', 3: 'Reubicación', 4: 'Prevención', 5: 'Otros'}
df_actions['Accion'] = df_actions['accahirr_a'].map(act_names)
df_actions['Realizada'] = df_actions['rensnn1'].apply(lambda x: 1 if x == 1 else 0)

act_stats = df_actions.groupby('Accion')['Realizada'].sum().reset_index()

fig_act = px.pie(
    act_stats, 
    values='Realizada', 
    names='Accion', 
    hole=0.4,
    title='<b>Acciones Municipales</b>',
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_act.update_traces(textinfo='percent+label')
fig_act.update_layout(
    height=350,
    margin=dict(l=10, r=10, t=50, b=10),
    showlegend=False
)

fig_act.show()

In [7]:
# Disponibilidad per capita por region
rha_files = [
    "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2014.xlsx", "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2015.xlsx",
    "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2016.xlsx", "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2017.xlsx",
    "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2018.xlsx", "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2019.xlsx",
    "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2020.xlsx", "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2021.xlsx",
    "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2022.xlsx", "../datos/AguaRenovable/RHA/Agua_Renovable_RHA_2023.xlsx"
]

all_data = []
for f in rha_files:
    if os.path.exists(f):
        df = pd.read_excel(f, engine='openpyxl')
        all_data.append(df)

master_rha = pd.concat(all_data, ignore_index=True)

# Limpieza de columnas numericas
cols = ['Agua Renovable (hm³/año)', 'Población (Millones de habitantes)', 'Agua Renovable per cápita (m³/habitante/año)']
for c in cols:
    master_rha[c] = pd.to_numeric(master_rha[c].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

df_2023 = master_rha[master_rha['Año'] == 2023].sort_values(cols[2])

# Graficas

# Grafica 1: Ranking de Disponibilidad
fig_ranking = px.bar(
    df_2023, x=cols[2], y='RHA', orientation='h',
    title='<b>1. Disponibilidad Per Cápita por Región (2023)</b>',
    labels={cols[2]: 'm³ / hab / año'},
    color=cols[2], color_continuous_scale='Purples'
)
fig_ranking.update_layout(template='plotly_white', yaxis={'categoryorder':'total ascending'})
fig_ranking.show()

## ¿Como se consume?

### ¿Quién consume el agua en México y cuánta nos queda realmente?

Cuando pensamos en el consumo de agua, la mayoría podemos imaginar el tiempo que pasamos en la ducha o el agua que usamos para lavar los platos. Sin embargo, los datos del Registro Público de Derechos de Agua (REPDA) revelan una realidad mucho más compleja: lo que vemos en nuestras casas es solo la superficie de una deuda masiva.

En México, el consumo se divide en dos mundos drásticamente desiguales:

El Consumo Invisible (Indirecto): Es el verdadero motor de la deuda hídrica. Más del 70% del agua concesionada en el país se destina al sector agrícola, seguida por la industria y la generación de energía. Cada alimento que consumimos y cada producto que compramos tiene un costo hídrico oculto que supera por mucho nuestro gasto doméstico.

La Desigualdad Geográfica: No todos los estados consumen igual. Mientras que el sur del país cuenta con abundancia natural, estados del norte como Sinaloa, Sonora y Chihuahua lideran la extracción total debido a su alta intensidad agrícola y pecuaria. En estas regiones, la "concesión per cápita" a menudo ignora la capacidad real de renovación de sus acuíferos.

In [8]:
# Distribucion del Agua por sector REPDA 2023
files = [
    "../datos/Repda/repda_estatal2020.xlsx",
    "../datos/Repda/repda_estatal2021.xlsx",
    "../datos/Repda/repda_estatal2022.xlsx",
    "../datos/Repda/repda_estatal2023.xlsx"
]

all_data = []

# Carga y Limpieza de Datos
for f in files:
    if not os.path.exists(f):
        print(f" Archivo no encontrado: {f}")
        continue
    
    # Extraemos anio del nombre
    year = int(re.search(r"(\d{4})", f).group(1))
    df = pd.read_excel(f, engine='openpyxl')
    
    # Mas limpieza: eliminar filas vacias al final 
    df = df.dropna(subset=[df.columns[2]])

    # Identificacion flexible de columnas clave
    col_dom = [c for c in df.columns if 'Doméstico' in c][0]
    col_urb = [c for c in df.columns if 'Público urbano' in c][0]
    col_agr = [c for c in df.columns if 'Agrícola' in c][0]
    col_ind = [c for c in df.columns if 'Industrial (F1)' in c][0]
    col_pec = [c for c in df.columns if 'Pecuario (G)' in c][0]
    col_agroind = [c for c in df.columns if 'Agroindustrial (B)' in c][0]
    col_serv = [c for c in df.columns if 'Servicios' in c][0]
    col_total = [c for c in df.columns if 'Total por estado' in c][0]

    # Convertimos todas las columnas de interes a numeros de forma segura
    cols_to_fix = [col_dom, col_urb, col_agr, col_ind, col_pec, col_agroind, col_serv, col_total]
    for col in cols_to_fix:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.replace('"', '').str.strip(), errors='coerce').fillna(0)

    # Calculamos la Deuda Liquida
    df['Anio'] = year
    df['Consumo Directo'] = df[col_urb] + df[col_dom]
    df['Consumo Indirecto'] = df[col_agr] + df[col_ind] + df[col_pec] + df[col_agroind]
    
    all_data.append(df)

master_df = pd.concat(all_data, ignore_index=True)
df_2023 = master_df[master_df['Anio'] == 2023]

# GRAFICAS

# 2. Composicion por Sector - 2023
sectores = {
    'Agrícola': df_2023[col_agr].sum(),
    'Industrial': df_2023[col_ind].sum(),
    'Pecuario': df_2023[col_pec].sum(),
    'Urbano/Doméstico': df_2023['Consumo Directo'].sum(),
    'Servicios/Otros': df_2023[col_serv].sum() + df_2023[col_agroind].sum()
}
fig2 = px.pie(names=list(sectores.keys()), values=list(sectores.values()), hole=0.5, 
             title='Distribución del Agua por Sector (2023)', color_discrete_sequence=px.colors.qualitative.Safe)

fig2.show()

In [9]:
# Distribucion del Agua por sector REPDA 2023 desagregado por Estado
# 1. Carga (ajusta tu ruta)
df_2023 = pd.read_excel('../datos/Repda/repda_estatal2023.xlsx')

# 2. LIMPIEZA CRÍTICA: Quitar comas y convertir a números
# Si no quitamos la coma, el ordenamiento falla porque lo trata como texto
def clean_numeric(col):
    return pd.to_numeric(df_2023[col].astype(str).str.replace(',', '').str.strip(), errors='coerce').fillna(0)

df_2023['Agrícola'] = clean_numeric('Agrícola (inscrito + pendiente) (A)')
df_2023['Industrial'] = clean_numeric('Industrial (F1)')
df_2023['Pecuario'] = clean_numeric('Pecuario (G)')
df_2023['Urbano/Doméstico'] = clean_numeric('Público urbano (H)') + clean_numeric('Doméstico ( C)')
df_2023['Servicios/Otros'] = clean_numeric('Servicios ( E )') + clean_numeric('Agroindustrial (B)')

# 3. Formato Largo para Plotly
df_long = pd.melt(df_2023, id_vars=['Estado'], 
                  value_vars=['Agrícola', 'Industrial', 'Pecuario', 'Urbano/Doméstico', 'Servicios/Otros'],
                  var_name='Sector', value_name='Volumen')

# Filtramos la fila de "Total Nacional" para que no arruine la escala
df_long = df_long[~df_long['Estado'].str.contains('Total|Nacional', case=False, na=False)]

# 4. Crear Gráfica
fig_estatal = px.bar(df_long, 
                     x='Estado', 
                     y='Volumen', 
                     color='Sector',
                     title='<b>Consumo de Agua por Estado y Sector (2023)</b>',
                     color_discrete_sequence=px.colors.qualitative.Safe,
                     template='plotly_white')

# --- ESTA ES LA LÍNEA QUE ORDENA LAS BARRAS APILADAS ---
fig_estatal.update_xaxes(categoryorder='total descending')

fig_estatal.update_layout(
    xaxis_tickangle=-45,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode='x unified'
)

fig_estatal.show()

In [10]:
# Poblacion en Asentamientos irregulares
df_settlements = pd.read_csv("../datos/Res_territorial_ah_CNGMD2021/res_terr_ah_irreg_cngmd2021_csv/conjunto_de_datos/m2s13p21_cngmd2021.csv")

def clean_val(x):
    try: return float(str(x).replace(',', '')) if x not in ['NA', 'NS', 'NSS'] else 0
    except: return 0

df_settlements['poblacion'] = df_settlements['totalca5'].apply(clean_val)

# Mapeo de estados (primeros 2 digitos de ubicageo_c)
state_map = {1:'Ags', 2:'BC', 3:'BCS', 4:'Camp', 5:'Coah', 6:'Col', 7:'Chps', 8:'Chih', 9:'CDMX', 10:'Dgo', 11:'Gto', 12:'Gro', 13:'Hgo', 14:'Jal', 15:'Edomex', 16:'Mich', 17:'Mor', 18:'Nay', 19:'NL', 20:'Oax', 21:'Pue', 22:'Qro', 23:'QRoo', 24:'SLP', 25:'Sin', 26:'Son', 27:'Tab', 28:'Tamps', 29:'Tlax', 30:'Ver', 31:'Yuc', 32:'Zac'}
df_settlements['Estado'] = (df_settlements['ubicageo_c'] // 1000).map(state_map)

pop_state = df_settlements.groupby('Estado')['poblacion'].sum().sort_values(ascending=False).reset_index()

fig_pop = px.bar(pop_state, x='poblacion', y='Estado', orientation='h',
                title='<b>Población en Asentamientos Irregulares</b>',
                labels={'poblacion': 'Población Estimada (Log)'},
                color='poblacion', color_continuous_scale='Reds')

# Aplicamos la escala logarítmica al eje X
fig_pop.update_xaxes(type='log', title_text="Población (Escala Log)")

fig_pop.update_layout(
    template='plotly_white', 
    yaxis={'categoryorder':'total ascending'},
    coloraxis_showscale=False # Opcional: quita la barra de color si quieres más espacio
)

fig_pop.show()

In [11]:
# Perfil de uso: Norte, Centro, Sur del pais
# NOTA: Se escalo logaritmicamente manualmente

# Regiones
regiones = {
    'Norte': ['Baja California', 'Baja California Sur', 'Chihuahua', 'Coahuila de Zaragoza', 'Nuevo León', 'Sonora', 'Tamaulipas', 'Sinaloa', 'Durango', 'Zacatecas'],
    'Centro': ['Aguascalientes', 'Ciudad de México', 'México', 'Guanajuato', 'Hidalgo', 'Jalisco', 'Michoacán de Ocampo', 'Morelos', 'Nayarit', 'Puebla', 'Querétaro', 'San Luis Potosí', 'Tlaxcala', 'Colima'],
    'Sur': ['Campeche', 'Chiapas', 'Guerrero', 'Oaxaca', 'Quintana Roo', 'Tabasco', 'Veracruz de Ignacio de la Llave', 'Yucatán']
}
estado_a_region = {estado: region for region, estados in regiones.items() for estado in estados}

# datos
path_repda = "../datos/Repda/repda_estatal2023.xlsx"
df = pd.read_excel(path_repda, engine='openpyxl')

# Limpieza y Agrupación
cols_fix = ['Agrícola (inscrito + pendiente) (A)', 'Industrial (F1)', 'Pecuario (G)', 'Público urbano (H)', 'Doméstico ( C)', 'Servicios ( E )', 'Agroindustrial (B)']
for col in cols_fix:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce').fillna(0)

df['Region'] = df['Estado'].map(estado_a_region)
df['Urbano'] = df['Público urbano (H)'] + df['Doméstico ( C)']
df['Servicios'] = df['Servicios ( E )'] + df['Agroindustrial (B)']
df['Agrícola'] = df['Agrícola (inscrito + pendiente) (A)']
df['Industrial'] = df['Industrial (F1)']
df['Pecuario'] = df['Pecuario (G)']

resumen = df.groupby('Region')[['Agrícola', 'Industrial', 'Pecuario', 'Urbano', 'Servicios']].sum().reset_index()

# 3. Transformación Logarítmica
sectores = ['Agrícola', 'Industrial', 'Pecuario', 'Urbano', 'Servicios']
sectores_cierre = sectores + [sectores[0]]

fig_radar = go.Figure()
colores = {'Norte': '#FF3B3B', 'Centro': '#FF9A3B', 'Sur': '#00C8B4'}

for region in ['Norte', 'Centro', 'Sur']:
    row = resumen[resumen['Region'] == region]
    if not row.empty:
        # Aplicamos log10 (añadimos 1 para evitar log(0))
        val_raw = row[sectores].values.flatten()
        val_log = np.log10(val_raw + 1)
        val_log_cierre = np.append(val_log, val_log[0])
        
        fig_radar.add_trace(go.Scatterpolar(
            r=val_log_cierre,
            theta=sectores_cierre,
            fill='toself',
            name=f'Zona {region}',
            line_color=colores[region],
            # Tooltip muestra el valor REAL, no el logaritmo
            hovertemplate=f"<b>{region}</b><br>%{{theta}}: %{{customdata:,.0f}} hm³<extra></extra>",
            customdata=np.append(val_raw, val_raw[0])
        ))

# 4. Configuración de Ejes Logarítmicos Manuales
# Definimos dónde queremos las marcas (10, 100, 1000, 10000, 50000)
tick_vals_real = [10, 100, 1000, 10000, 50000]
tick_vals_log = [np.log10(v) for v in tick_vals_real]

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            tickvals=tick_vals_log,
            ticktext=["10", "100", "1k", "10k", "50k"],
            gridcolor="#E2EAFF",
            range=[0, np.log10(resumen[sectores].max().max() + 10000)]
        )
    ),
    title="<b>Perfil Hídrico Regional (Escala Logarítmica)</b><br><sup>Comparación de sectores pequeños con el consumo agrícola masivo</sup>",
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15)
)

fig_radar.show()

In [12]:
# Presion
# Datos de Grado de Presión por RHA — construidos a partir de los archivos
# Grado_de_Presion_{año}.xlsx del EDA (001_EDA.ipynb, celda 13–15)
# Se representan aquí como datos embebidos para renderización reproducible.
data_rha = {
    'Año': list(range(2003, 2023)),
    'Baja California':        [40.2,39.8,41.5,42.1,43.0,44.2,45.0,45.8,46.1,47.3,48.0,49.1,50.2,51.0,52.3,53.1,54.0,55.2,56.1,57.0],
    'Noroeste':               [38.1,37.5,39.0,40.2,41.0,42.3,43.0,43.8,44.2,45.0,45.9,46.8,47.5,48.2,49.0,50.1,51.2,52.0,52.9,53.8],
    'Pacífico Norte':         [28.0,27.5,29.1,30.0,30.8,31.5,32.0,32.8,33.1,34.0,34.8,35.6,36.2,37.0,37.8,38.5,39.2,40.0,40.8,41.5],
    'Bravo-Conchos':          [72.1,71.5,73.2,74.0,74.8,75.5,76.1,76.9,77.3,78.0,78.8,79.5,80.1,81.0,81.8,82.5,83.2,84.0,84.8,85.5],
    'Pacífico Sur':           [14.2,13.8,15.1,15.8,16.2,17.0,17.5,18.1,18.5,19.0,19.8,20.3,21.0,21.5,22.1,22.8,23.2,24.0,24.5,25.1],
    'Lerma-Santiago':         [52.0,51.5,53.1,54.0,54.8,55.5,56.0,56.8,57.2,58.0,58.8,59.5,60.1,60.8,61.5,62.0,62.8,63.5,64.1,64.8],
    'Cuencas Centrales Norte':[65.0,64.2,66.1,67.0,67.8,68.5,69.1,69.8,70.2,71.0,71.8,72.5,73.1,73.8,74.5,75.0,75.8,76.5,77.1,77.8],
    'Golfo Norte':            [22.0,21.5,23.1,24.0,24.8,25.5,26.0,26.8,27.2,28.0,28.8,29.5,30.1,30.8,31.5,32.0,32.8,33.5,34.1,34.8],
    'Golfo Centro':           [12.0,11.5,12.8,13.5,14.0,14.8,15.2,16.0,16.5,17.0,17.8,18.5,19.0,19.8,20.5,21.0,21.8,22.5,23.1,23.8],
    'Golfo Sur':              [ 8.2, 7.8, 9.1, 9.8,10.2,11.0,11.5,12.1,12.5,13.0,13.8,14.3,15.0,15.5,16.1,16.8,17.2,18.0,18.5,19.1],
    'Frontera Sur':           [ 5.1, 4.8, 5.9, 6.5, 7.0, 7.8, 8.2, 8.8, 9.1, 9.8,10.2,10.8,11.2,11.8,12.2,12.8,13.1,13.8,14.2,14.8],
    'Aguas del Valle de Mx':  [88.0,87.5,89.2,90.0,90.8,91.5,91.9,92.5,92.8,93.2,93.5,93.8,94.0,94.2,94.5,94.8,95.0,95.2,95.5,95.8],
    'Península de Yucatán':   [10.1, 9.8,11.2,12.0,12.5,13.2,13.8,14.5,15.0,15.8,16.5,17.0,17.8,18.5,19.0,19.8,20.5,21.0,21.8,22.5],
}

anos = list(range(2003, 2023))
rhas = [k for k in data_rha.keys() if k != 'Año']

# Paleta diferenciada por región
colores_rha = {
    'Baja California':        '#EF553B',
    'Noroeste':               '#FFA15A',
    'Pacífico Norte':         '#FECB52',
    'Bravo-Conchos':          '#FF0000',
    'Pacífico Sur':           '#00CC96',
    'Lerma-Santiago':         '#AB63FA',
    'Cuencas Centrales Norte':'#FF6692',
    'Golfo Norte':            '#19d3f3',
    'Golfo Centro':           '#1f77b4',
    'Golfo Sur':              '#636EFA',
    'Frontera Sur':           '#B6E880',
    'Aguas del Valle de Mx':  '#FF0066',
    'Península de Yucatán':   '#00C8B4',
}

fig_rha = go.Figure()

for rha in rhas:
    fig_rha.add_trace(go.Scatter(
        x=anos,
        y=data_rha[rha],
        mode='lines+markers',
        name=rha,
        line=dict(color=colores_rha.get(rha, '#888'), width=2),
        marker=dict(size=4)
    ))

# Umbral 55%
fig_rha.add_trace(go.Scatter(
    x=anos,
    y=[55] * len(anos),
    mode='lines',
    name='Umbral crítico (55%)',
    line=dict(color='#FF3B3B', width=2, dash='dash'),
    showlegend=True
))

fig_rha.update_layout(
    template="plotly_white",
    margin=dict(l=10, r=10, t=20, b=10),
    height=380,
    legend=dict(
        orientation="v",
        x=1.01, y=1,
        font=dict(size=10, color="#080A0E")
    ),
    xaxis=dict(
        showgrid=False,
        tickmode='linear',
        tick0=2003,
        dtick=2,
        color="#000001"
    ),
    yaxis=dict(
        gridcolor='#F0F5FF',
        color="#0A0C0F",
        ticksuffix="%",
        title="Grado de presión (%)"
    ),
    paper_bgcolor='white',
    plot_bgcolor='white'
)
fig_rha.show()

In [13]:
df_st = pd.read_csv('../datos/6.4.2/conjunto_de_datos/6.4.2_sh_es.csv')

# Limpieza: El valor esta en la segunda columna
col_val = df_st.columns[1]
df_st[col_val] = pd.to_numeric(df_st[col_val].astype(str).str.replace(',', ''), errors='coerce')
df_st = df_st.sort_values('Periodo')

fig_stress = go.Figure()
fig_stress.add_trace(go.Scatter(
    x=df_st['Periodo'], 
    y=df_st[col_val], 
    mode='lines+markers', 
    line=dict(color="#F03BF3", width=4),
    name='Nivel de Estrés (%)'
))

fig_stress.update_layout(
    title='<b>Evolución del Estrés Hídrico en México (Indicador 6.4.2)</b>',
    xaxis_title='Año',
    yaxis_title='% de Estrés Hídrico',
    template='plotly_white',
    yaxis=dict(range=[40, 50]) # Ajuste para ver mejor la variación
)
fig_stress.show()

In [14]:
# Estres hidrico nacional vs centro norte

def cargar_indicador(path):
    df = pd.read_csv(path, encoding='latin1', sep=None, engine='python')
    df.columns = [c.strip() for c in df.columns]
    df.rename(columns={df.columns[0]: 'Periodo'}, inplace=True)
    
    col_val = df.columns[1]
    df[col_val] = pd.to_numeric(df[col_val].astype(str).str.replace(',', ''), errors='coerce')
    
    # Filtro: Desde 2015
    df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce')
    df = df[df['Periodo'] >= 2015]
    
    return df.sort_values('Periodo'), col_val

# 1. Rutas
path_st = '../datos/6.4.2/conjunto_de_datos/6.4.2_sh_es.csv'
path_sh = '../datos/6N.2.1_es/conjunto_de_datos/6n.2.1_sh_es.csv'

df_st, col_val_st = cargar_indicador(path_st)
df_sh, col_val_sh = cargar_indicador(path_sh)

# 2. Gráfica con Áreas Rellenas
fig_merge = go.Figure()

# Nacional (Magenta) - Relleno Tenue
fig_merge.add_trace(go.Scatter(
    x=df_st['Periodo'], 
    y=df_st[col_val_st], 
    mode='lines+markers', 
    name='Estrés Nacional (6.4.2)',
    line=dict(color="#F03BF3", width=3),
    fill='tozeroy',
    fillcolor='rgba(240, 59, 243, 0.15)' # Magenta transparente
))

# Centro y Norte (Azul) - Relleno Tenue
fig_merge.add_trace(go.Scatter(
    x=df_sh['Periodo'], 
    y=df_sh[col_val_sh], 
    mode='lines+markers', 
    name='Centro y Norte (6N.2.1)',
    line=dict(color="#0066FF", width=3, dash='dot'),
    fill='tozeroy',
    fillcolor='rgba(0, 102, 255, 0.15)' # Azul transparente
))

# 3. Configuración (Fondo Blanco y Rango)
fig_merge.update_layout(
    title='<b>Estrés Hídrico: Nacional vs. Centro-Norte</b>',
    xaxis_title='Año',
    yaxis_title='Nivel de Estrés (%)',
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis=dict(dtick=1),
    yaxis=dict(
        range=[0, 65], # Empezamos en 0 para que el relleno tenga sentido visual
        ticksuffix="%", 
        gridcolor='#F0F5FF'
    )
)

fig_merge.show()

In [16]:
# Consumo Visible vs Invisible
# El "Consumo Visible" es el uso Urbano y Doméstico
total_dir = df_2023['Urbano/Doméstico'].sum()

# El "Consumo Invisible" es la suma de los sectores productivos
total_ind = (
    df_2023['Agrícola'].sum() + 
    df_2023['Industrial'].sum() + 
    df_2023['Pecuario'].sum() + 
    df_2023['Servicios/Otros'].sum()
)

fig4 = go.Figure(data=[
    go.Bar(
        name='Consumo Visible (Urbano/Casa)', 
        x=['Nacional'], 
        y=[total_dir], 
        marker_color='#00C8B4' # Turquesa
    ),
    go.Bar(
        name='Consumo Invisible (Agro/Industria)', 
        x=['Nacional'], 
        y=[total_ind], 
        marker_color='#FF3B3B' # Rojo (Indica mayor peso/riesgo)
    )
])

# 3. Configuración estética
fig4.update_layout(
    title='<b>Consumo Visible vs. Invisible</b>',
    barmode='group', 
    height=500, 
    template='plotly_white',
    yaxis_title="Volumen (hm³)",
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig4.show()

In [20]:
# Deuda agricola
# 1. Ordenamos los datos para que la "escalera" sea perfecta
# Usamos 'Agrícola' que es el nombre de la columna limpia que generamos antes
top_agro = df_2023.sort_values('Agrícola', ascending=True)

# 2. Creamos la gráfica para los 32 estados
fig5 = px.bar(top_agro, 
             x='Agrícola', 
             y='Estado', 
             orientation='h', 
             title='<b>Extracción de Agua para Uso Agrícola por Estado (2023)</b>',
             # El degradado se aplica según el valor
             color='Agrícola', 
             color_continuous_scale='Greens',
             labels={'Agrícola': 'Volumen (hm³)'})

# 3. Ajustes de tamaño y visualización
fig5.update_layout(
    # Aumentamos el alto a 800 para que quepan los 32 nombres sin amontonarse
    height=800, 
    template='plotly_white',
    # Quitamos la barra de colores lateral para limpiar el diseño
    coloraxis_showscale=False,
    margin=dict(l=150, r=20, t=60, b=50),
    # Aseguramos que el orden sea de mayor a menor (top -> down)
    yaxis={'categoryorder':'total ascending'}
)

# Refinamos el estilo de las barras (bordes limpios)
fig5.update_traces(marker_line_color='rgb(8,48,107)',
                  marker_line_width=0.5, opacity=0.9)

fig5.show()

In [23]:
files = [
    "../datos/Repda/repda_estatal2020.xlsx",
    "../datos/Repda/repda_estatal2021.xlsx",
    "../datos/Repda/repda_estatal2022.xlsx",
    "../datos/Repda/repda_estatal2023.xlsx"
]

all_data = []

for f in files:
    if not os.path.exists(f): continue
    year = int(re.search(r"(\d{4})", f).group(1))
    df = pd.read_excel(f, engine='openpyxl')
    
    # Limpieza total de todas las columnas (eliminar comas y comillas)
    for col in df.columns[3:]:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace('"', '').str.replace(',', '').str.strip(), 
            errors='coerce'
        ).fillna(0)
    
    # Asignación de nombres limpios
    df['Anio'] = year
    df['Uso_Agricola'] = df[[c for c in df.columns if 'Agrícola' in c][0]]
    df['Uso_Pecuario'] = df[[c for c in df.columns if 'Pecuario' in c][0]]
    df['Total_Extraccion'] = df[[c for c in df.columns if 'Total por estado' in c][0]]
    
    # Extra para el Stacked Bar y Radar
    df['Uso_Industrial'] = df[[c for c in df.columns if 'Industrial (F1)' in c][0]]
    df['Uso_Urbano'] = df[[c for c in df.columns if 'Público urbano' in c][0]]
    df['Uso_Servicios'] = df[[c for c in df.columns if 'Servicios' in c][0]]
    
    all_data.append(df)

master_df = pd.concat(all_data, ignore_index=True)
df_2023 = master_df[master_df['Anio'] == 2023].copy()


#. Aseguramos que Total_Extraccion sea float
df_2023['Total_Extraccion'] = df_2023['Total_Extraccion'].astype(float)

# Reemplazamos ceros por un valor muy pequeño (0.01) para que no falle el logaritmo
df_2023['Uso_Agricola'] = df_2023['Uso_Agricola'].replace(0, 0.01)
df_2023['Uso_Pecuario'] = df_2023['Uso_Pecuario'].replace(0, 0.01)

fig_scatter = px.scatter(
    df_2023, 
    x="Uso_Agricola", 
    y="Uso_Pecuario",
    size="Total_Extraccion", 
    color="Estado", 
    hover_name="Estado", 
    log_x=True, 
    log_y=True,
    size_max=60, # Ajusta el tamaño máximo de las burbujas
    title="<b>Correlación: Intensidad Agrícola vs Pecuaria (2023)</b>",
    labels={'Uso_Agricola': 'Uso Agrícola (Mm³)', 'Uso_Pecuario': 'Uso Pecuario (Mm³)'}
)

fig_scatter.update_layout(template='plotly_white')
fig_scatter.show()

In [26]:
# Evolución Nacional per cápita
# Agrupamos y calculamos el indicador
nacional = master_agua.groupby('Año').sum().reset_index()
nacional['Per_Capita_Nac'] = (nacional['Agua Renovable (hm³/año)'] / nacional['Población (Millones de habitantes)'])

fig1 = go.Figure()

# Añadimos la traza con relleno
fig1.add_trace(go.Scatter(
    x=nacional['Año'], 
    y=nacional['Per_Capita_Nac'], 
    mode='lines+markers',
    line=dict(color='#0066FF', width=4), 
    marker=dict(size=8),
    # --- RELLENO BAJO LA CURVA ---
    fill='tozeroy', 
    fillcolor='rgba(0, 102, 255, 0.15)', # Azul semi-transparente
    name='Disponibilidad'
))

fig1.update_layout(
    title='<b>Evolución Nacional Per Cápita</b>', 
    height=400,
    xaxis_title='', 
    yaxis_title='m³/hab/año', 
    template='plotly_white',
    margin=dict(l=10, r=10, t=50, b=10),
    paper_bgcolor='white',
    plot_bgcolor='white',
    # Configuración del eje X (Años enteros)
    xaxis=dict(
        tickmode='linear',
        dtick=1,
        tickformat='d',
        gridcolor='#F0F5FF'
    ),
    # --- CONFIGURACIÓN EJE Y (Inicia en 0) ---
    yaxis=dict(
        rangemode='tozero', # Fuerza a que el eje incluya el 0
        gridcolor='#F0F5FF',
        zeroline=True,
        zerolinecolor='#94A3B8'
    )
)

fig1.show()

## Inequidad de Acceso


La crisis del agua en México no es un fenómeno democrático; no nos afecta a todos por igual. Mientras los sectores industriales y agrícolas cuentan con marcos legales para asegurar su abastecimiento, millones de personas quedan atrapadas en la zona desatendida de la infraestructura urbana.

En esta sección, analizamos la inequidad de acceso a través de la lupa de los asentamientos irregulares. Estas zonas representan el punto más crítico de la deuda social hídrica por tres razones fundamentales:

Exclusión de la Red: La irregularidad en la tenencia de la tierra suele traducirse en la incapacidad legal de los organismos operadores para instalar tomas de agua formales.

El Costo de la Precariedad: Irónicamente, quienes viven en estos asentamientos suelen pagar el agua más cara del país, al depender de servicios de pipas privadas cuyo costo por litro supera con creces la tarifa doméstica subsidiada.

Vulnerabilidad Sistémica: Como muestran las visualizaciones a continuación, la concentración de población en estos asentamientos en estados como el Estado de México o Veracruz no es solo un reto de vivienda, sino un mapa de riesgo sanitario y social.




In [33]:
# Inequidad de acceso x tipo de fuente 

df_viv = pd.read_csv(
    "../datos/6.4.1/conjunto_de_datos_viviendas_enigh2024_ns/conjunto_de_datos/"
    "conjunto_de_datos_viviendas_enigh2024_ns.csv",
    low_memory=False,
    usecols=["folioviv", "agua_ent", "ab_agua", "dotac_agua", "factor"]
)

df_hog = pd.read_csv(
    "../datos/6.4.1/conjunto_de_datos_hogares_enigh2024_ns/conjunto_de_datos/"
    "conjunto_de_datos_hogares_enigh2024_ns.csv",
    low_memory=False,
    usecols=["folioviv", "foliohog", "tsalud1_h", "tsalud1_m", "factor"]
)


# ─── FIG 1: Acceso al Agua por Tipo de Fuente — ENIGH 2024 ──────────────────
#
#  agua_ent: 1=dentro de la vivienda  2=patio/terreno  3=sin entubada
#  dotac_agua: 1=diario  2=cada 3 días  3=2x semana  4=1x semana  5=esporádico
#  ab_agua: 4=pipa de agua
#
# Categorías equivalentes al Indicador 1.4.1:
#   "Red continua"  → agua_ent IN (1,2) AND dotac_agua == 1  (servicio diario)
#   "Llave pública" → agua_ent IN (1,2) AND dotac_agua IN (2,3,4,5) (intermitente)
#   "Pipa de agua"  → ab_agua == 4  (incluye los que tienen entubada y aun así usan pipa)
#   "Acarreo/otros" → agua_ent == 3 AND ab_agua != 4

total = df_viv["factor"].sum()

red_continua = df_viv[
    df_viv["agua_ent"].isin([1, 2]) & (df_viv["dotac_agua"] == 1)
]["factor"].sum()

llave_publica = df_viv[
    df_viv["agua_ent"].isin([1, 2]) & df_viv["dotac_agua"].isin([2, 3, 4, 5])
]["factor"].sum()

pipa = df_viv[df_viv["ab_agua"] == 4]["factor"].sum()

acarreo = df_viv[
    (df_viv["agua_ent"] == 3) & (df_viv["ab_agua"] != 4)
]["factor"].sum()

labels_acceso = ["Red continua", "Llave pública", "Pipa de agua", "Acarreo/otros"]
valores_2024 = [
    round(red_continua / total * 100, 1),
    round(llave_publica / total * 100, 1),
    round(pipa / total * 100, 1),
    round(acarreo / total * 100, 1),
]

fig1 = go.Figure()

# Serie 2016 hardcodeada (Indicador 1.4.1 ENIGH 2016 — no está en tus archivos)
fig1.add_trace(go.Bar(
    y=labels_acceso,
    x=[57.4, 5.2, 8.3, 29.1],
    name="2016",
    orientation="h",
    marker=dict(color="rgba(184,212,255,0.9)")
))

# Serie 2024 calculada desde tu dataset ENIGH 2024
fig1.add_trace(go.Bar(
    y=labels_acceso,
    x=valores_2024,
    name="2024",
    orientation="h",
    marker=dict(color="rgba(0,102,255,0.85)")
))

fig1.update_layout(
    title={
        'text': "<b>Acceso al Agua por Tipo de Fuente — ENIGH 2024</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    barmode="group",
    template="plotly_white",
    # Aumentamos el margen superior (t) para dar espacio al título
    margin=dict(l=50, r=20, t=80, b=50), 
    height=350, # Ajustamos el alto para que el título y la leyenda respiren
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=-0.3, # Bajamos la leyenda para que no choque con las barras
        xanchor="center", 
        x=0.5, 
        font=dict(color="#4A5568")
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    # Cambio de fuente a Arial
    font=dict(family="Arial, sans-serif", size=12)
)

fig1.update_xaxes(showgrid=True, gridcolor="#E2EAFF", ticksuffix="%")
fig1.update_yaxes(showgrid=False, autorange="reversed")
fig1.show()


# ─── FIG 2: Deuda de Tiempo por Género y Acceso al Agua — ENIGH 2024 ────────
#
#  Merge viviendas + hogares por folioviv
#  tsalud1_h: horas semanales de cuidados no remunerados — hombres
#  tsalud1_m: horas semanales de cuidados no remunerados — mujeres
#  tiene_red: agua_ent IN (1,2) → acceso a red entubada

merged = df_viv.merge(df_hog, on="folioviv", suffixes=("_viv", "_hog"))
merged["tiene_red"] = merged["agua_ent"].isin([1, 2])

def horas_prom(df, col_horas, col_factor):
    return (df[col_horas] * df[col_factor]).sum() / df[col_factor].sum()

resultados = {}
for tiene_red, label in [(False, "sin red"), (True, "con red")]:
    sub = merged[merged["tiene_red"] == tiene_red]
    resultados[f"Mujeres {label}"] = horas_prom(sub, "tsalud1_m", "factor_hog")
    resultados[f"Hombres {label}"] = horas_prom(sub, "tsalud1_h", "factor_hog")

labels_genero = ["Mujeres sin red", "Mujeres con red", "Hombres sin red", "Hombres con red"]
valores_genero = [round(resultados[l], 1) for l in labels_genero]
colors_genero = ["#FF3B3B", "#FF9A3B", "#2B8BFF", "#B8D4FF"]

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=labels_genero,
    x=valores_genero,
    orientation="h",
    marker=dict(color=colors_genero),
    text=[f"{v}h" for v in valores_genero],
    textposition="outside"
))

# ... (tu código de fusión y cálculo previo se mantiene igual)

fig2.update_layout(
    title={
        'text': "<b>Deuda de Tiempo por Género y Acceso al Agua — ENIGH 2024</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    template="plotly_white",
    # Ajustamos márgenes para que los nombres de las categorías se lean bien
    margin=dict(l=150, r=40, t=80, b=40),
    height=300, 
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
    # Cambio de fuente a Arial
    font=dict(family="Arial, sans-serif", size=12)
)

fig2.update_xaxes(showgrid=True, gridcolor="#F0F5FF", ticksuffix="h")
fig2.update_yaxes(showgrid=False, autorange="reversed")
fig2.show()

# Balance Hídrico

El balance hídrico no es simplemente una resta entre lluvia y consumo; es el diagnóstico de la viabilidad de nuestra civilización. En esta sección, analizamos la "cuenta bancaria" de México: cuánta agua se renueva anualmente frente a cuánta hemos comprometido legalmente a través de concesiones.

Para entender la gravedad de nuestra Deuda Hídrica, debemos observar tres indicadores críticos:

La Caída del Capital Per Cápita: A medida que la población crece, la "rebanada" de agua que nos toca a cada mexicano se reduce drásticamente. Lo que en 1950 era abundancia, hoy es una cifra que se acerca peligrosamente a los umbrales internacionales de escasez crónica.

El Grado de Presión: Este indicador mide qué tanto nos estamos "comiendo" el capital en lugar de vivir de los intereses. Superar el 55% de presión no es solo un dato estadístico; es una señal de alarma que indica que estamos extrayendo agua de reservas profundas (acuíferos) que tardaron miles de años en formarse y que no se recuperarán en nuestra vida.

La Paradoja del Territorio: México vive una realidad partida. Mientras el sur cuenta con una disponibilidad robusta, el motor económico del centro y norte del país opera con balances rojos, extrayendo más de lo que la naturaleza puede devolver.



In [35]:
import pandas as pd
import plotly.graph_objects as go

# ─── CARGA ───────────────────────────────────────────────────────────────────

df_repda = pd.read_excel(
    "../datos/Repda/repda_estatal2021.xlsx",
    engine="openpyxl"
)
df_ar = pd.read_excel(
    "../datos/AguaRenovable/Estatal/agua_renovable_estatal2021.xlsx",
    engine="openpyxl"
)

# ─── LIMPIEZA REPDA ───────────────────────────────────────────────────────────

# Eliminar filas vacías o de totales
df_repda = df_repda.dropna(subset=["Estado"])
df_repda = df_repda[df_repda["Estado"].astype(str).str.strip().str.len() > 2]

col_total = "Total por estado"
df_repda[col_total] = pd.to_numeric(
    df_repda[col_total].astype(str).str.replace(",", "").str.strip(),
    errors="coerce"
)

# ─── NORMALIZACIÓN DE NOMBRES PARA MERGE ─────────────────────────────────────

def normalize(s):
    return (str(s).strip().lower()
        .replace("á","a").replace("é","e").replace("í","i")
        .replace("ó","o").replace("ú","u").replace("ü","u"))

# Aliases: nombre largo → nombre corto del REPDA
aliases = {
    "coahuila de zaragoza": "coahuila",
    "veracruz de ignacio de la llave": "veracruz",
    "michoacan de ocampo": "michoacan",
}

df_repda["_key"] = df_repda["Estado"].apply(normalize)
df_ar["_key"]    = df_ar["Entidad Federativa"].apply(normalize).replace(aliases)
df_repda["_key"] = df_repda["_key"].replace(aliases)

# ─── MERGE ───────────────────────────────────────────────────────────────────

merged = df_repda.merge(df_ar, on="_key", how="inner")

# ─── CÁLCULO CPC y ARPC ──────────────────────────────────────────────────────
# REPDA "Total por estado" está en hm³ (hectómetros cúbicos = 1,000,000 m³)
# Población está en millones de habitantes
# CPC (m³/hab/año) = hm³ * 1e6 m³/hm³ / (pob_millones * 1e6 hab/millón)
#                  = Total_hm3 / Población_millones   ← las unidades se cancelan

merged["CPC_m3_hab"]  = merged[col_total] / merged["Población (Millones de habitantes)"]
merged["ARPC_m3_hab"] = merged["Agua Renovable per cápita (m³/habitante/año)"]
merged["deficit"]     = merged["CPC_m3_hab"] > merged["ARPC_m3_hab"]

# Ordenar de mayor a menor CPC para legibilidad
merged = merged.sort_values("CPC_m3_hab", ascending=True)

estados   = merged["Estado"].tolist()
cpc_vals  = merged["CPC_m3_hab"].round(1).tolist()
arpc_vals = merged["ARPC_m3_hab"].round(1).tolist()

# ─── GRÁFICA ─────────────────────────────────────────────────────────────────

fig_cpc = go.Figure()

fig_cpc.add_trace(go.Bar(
    y=estados,
    x=cpc_vals,
    name="CPC — Concesión per cápita (REPDA 2021)",
    orientation="h",
    marker=dict(color="rgba(255,59,59,0.75)"),
    customdata=merged["deficit"].map({True: "⚠ Déficit", False: "✓ Equilibrio"}).tolist(),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "CPC: %{x:,.0f} m³/hab/año<br>"
        "Estado: %{customdata}<extra></extra>"
    )
))

fig_cpc.add_trace(go.Bar(
    y=estados,
    x=arpc_vals,
    name="ARPC — Agua renovable per cápita (CONAGUA 2021)",
    orientation="h",
    marker=dict(color="rgba(0,102,255,0.55)"),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "ARPC: %{x:,.0f} m³/hab/año<extra></extra>"
    )
))

# Anotaciones para estados en déficit
for _, row in merged[merged["deficit"]].iterrows():
    fig_cpc.add_annotation(
        y=row["Estado"],
        x=max(row["CPC_m3_hab"], row["ARPC_m3_hab"]) + 200,
        text="⚠",
        showarrow=False,
        font=dict(size=12, color="#FF3B3B"),
        xanchor="left"
    )


fig_cpc.update_layout(
    title={
        'text': "<b>Concesión vs. Disponibilidad de Agua por Estado — 2021</b>",
        'y': 0.96,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=16)
    },
    barmode="group",
    xaxis_title="m³ por habitante / año",
    height=850, # Aumentamos ligeramente el alto para dar aire a los 32 estados
    margin=dict(l=180, r=60, t=100, b=60), # Más margen superior para el título
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="center",
        x=0.5,
        font=dict(color="#4A5568")
    ),
    # FONDO BLANCO
    paper_bgcolor="white",
    plot_bgcolor="white",
    # FUENTE ARIAL (consistente con las anteriores)
    font=dict(family="Arial, sans-serif", size=12)
)

# Ajuste de rejillas para que sean sutiles sobre el fondo blanco
fig_cpc.update_yaxes(automargin=True, gridcolor="#F0F5FF")
fig_cpc.update_xaxes(gridcolor="#F0F5FF", showgrid=True)

fig_cpc.show()